# Phase 1 - Dashboard Data Preparation

Notebook ini bertujuan untuk menyiapkan dataset yang telah diproses sehingga dapat langsung digunakan oleh dashboard interaktif.

Tahapan yang dilakukan meliputi:

- Business KPI
- Sales Aggregation
- Product Aggregation
- Customer Aggregation
- Regional Aggregation

Output notebook ini berupa beberapa file CSV yang siap digunakan oleh aplikasi dashboard.

In [36]:
import pandas as pd
import numpy as np

from pathlib import Path

In [37]:
df = pd.read_csv(
    "Superstore_Clean.csv",
    encoding="latin1"
)

In [38]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

In [39]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Quarter"] = df["Order Date"].dt.quarter

In [40]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sales,Quantity,Discount,Profit,Year,Month,Month Name,Quarter,Day,Weekday
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.9600,2,0.00,41.9136,2016,11,November,4,8,Tuesday
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.9400,3,0.00,219.5820,2016,11,November,4,8,Tuesday
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.6200,2,0.00,6.8714,2016,6,June,2,12,Sunday
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,5,0.45,-383.0310,2015,10,October,4,11,Sunday
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.3680,2,0.20,2.5164,2015,10,October,4,11,Sunday


# Phase 2 — Business KPI

Pada tahap ini dihitung indikator utama bisnis (Key Performance Indicators) yang nantinya akan ditampilkan pada dashboard.

KPI yang dihitung meliputi:

- Total Sales
- Total Profit
- Total Orders
- Total Customers
- Average Order Value
- Profit Margin


In [41]:
total_sales = df["Sales"].sum()

total_profit = df["Profit"].sum()

total_orders = df["Order ID"].nunique()

total_customers = df["Customer ID"].nunique()

total_products = df["Product ID"].nunique()

average_order = total_sales / total_orders

profit_margin = (
    total_profit / total_sales
) * 100

kpi = pd.DataFrame({

    "Metric":[
        "Total Sales",
        "Total Profit",
        "Total Orders",
        "Total Customers",
        "Total Products",
        "Average Order Value",
        "Profit Margin"
    ],

    "Value":[
        total_sales,
        total_profit,
        total_orders,
        total_customers,
        total_products,
        average_order,
        profit_margin
    ]
})

kpi

,Metric,Value
0,Total Sales,2.297201e+06
1,Total Profit,2.863970e+05
2,Total Orders,5.009000e+03
3,Total Customers,7.930000e+02
4,Total Products,1.862000e+03
5,Average Order Value,4.586147e+02
6,Profit Margin,1.246722e+01


# Phase 3 — Dashboard Dataset Preparation

Pada tahap ini dibuat beberapa dataset agregasi yang akan digunakan langsung oleh dashboard.

Dataset dipisahkan berdasarkan kebutuhan visualisasi sehingga dashboard tidak perlu melakukan proses agregasi ulang setiap kali dijalankan.

Output dataset akan disimpan dalam folder `processed/`.

In [42]:
from pathlib import Path

OUTPUT_DIR = Path("processed")

OUTPUT_DIR.mkdir(exist_ok=True)

In [43]:
kpi.to_csv(
    OUTPUT_DIR / "kpi.csv",
    index=False
)

In [44]:
monthly_dashboard = (
    df.groupby(["Year","Month","Month Name"])
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Orders=("Order ID","nunique")
      )
      .reset_index()
)

monthly_dashboard.to_csv(
    OUTPUT_DIR / "monthly_sales.csv",
    index=False
)

In [45]:
category_dashboard = (
    df.groupby("Category")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Orders=("Order ID","nunique")
      )
      .reset_index()
)

category_dashboard.to_csv(
    OUTPUT_DIR / "category_sales.csv",
    index=False
)

In [46]:
region_dashboard = (
    df.groupby("Region")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
      .reset_index()
)

region_dashboard.to_csv(
    OUTPUT_DIR / "region_sales.csv",
    index=False
)

In [47]:
customer_dashboard = (
    df.groupby("Customer Name")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Orders=("Order ID","nunique")
      )
      .sort_values("Sales",ascending=False)
      .reset_index()
)

customer_dashboard.to_csv(
    OUTPUT_DIR / "customer_sales.csv",
    index=False
)

In [48]:
product_dashboard = (
    df.groupby("Product Name")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
      .sort_values("Sales",ascending=False)
      .reset_index()
)

product_dashboard.to_csv(
    OUTPUT_DIR / "product_sales.csv",
    index=False
)

In [49]:
import os

os.listdir("processed")

['region_performance.csv',
 'customer_sales.csv',
 'kpi.csv',
 'category_sales.csv',
 'customer_performance.csv',
 'product_sales.csv',
 'monthly_sales.csv',
 'region_sales.csv',
 'category_performance.csv',
 'sales_trend.csv',
 'dashboard_filters.json',
 'dashboard_kpi.csv',
 'product_performance.csv']

# Phase 4 — Dashboard KPI Dataset

Pada tahap ini dibuat dataset KPI yang akan digunakan sebagai ringkasan utama pada dashboard.

Dataset KPI berisi indikator bisnis yang akan ditampilkan pada bagian atas dashboard sehingga pengguna dapat langsung melihat performa bisnis secara keseluruhan.

In [50]:
dashboard_kpi = pd.DataFrame({

    "Total Sales":[df["Sales"].sum()],

    "Total Profit":[df["Profit"].sum()],

    "Total Orders":[df["Order ID"].nunique()],

    "Total Customers":[df["Customer ID"].nunique()],

    "Total Products":[df["Product ID"].nunique()],

    "Average Order Value":[
        df["Sales"].sum()/df["Order ID"].nunique()
    ],

    "Profit Margin (%)":[
        df["Profit"].sum()/df["Sales"].sum()*100
    ]

})

dashboard_kpi

,Total Sales,Total Profit,Total Orders,Total Customers,Total Products,Average Order Value,Profit Margin (%)
0,2.297201e+06,286397.0217,5009,793,1862,458.614666,12.467217


In [51]:
dashboard_kpi.to_csv(
    OUTPUT_DIR/"dashboard_kpi.csv",
    index=False
)

# Phase 5 — Dashboard Charts Dataset

Tahap ini menyiapkan dataset yang akan digunakan oleh grafik pada dashboard sehingga proses visualisasi menjadi lebih cepat dan efisien.

In [52]:
sales_trend = (
    df.groupby(["Year","Month","Month Name"])
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
      .reset_index()
)

sales_trend.to_csv(
    OUTPUT_DIR/"sales_trend.csv",
    index=False
)

In [53]:
category_performance = (
    df.groupby("Category")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Quantity=("Quantity","sum")
      )
      .reset_index()
)

category_performance.to_csv(
    OUTPUT_DIR/"category_performance.csv",
    index=False
)

In [54]:
region_performance = (
    df.groupby("Region")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
      .reset_index()
)

region_performance.to_csv(
    OUTPUT_DIR/"region_performance.csv",
    index=False
)

In [55]:
customer_performance = (
    df.groupby("Customer Name")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Orders=("Order ID","nunique")
      )
      .sort_values("Sales",ascending=False)
      .reset_index()
)

customer_performance.to_csv(
    OUTPUT_DIR/"customer_performance.csv",
    index=False
)

In [56]:
product_performance = (
    df.groupby("Product Name")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
      .sort_values("Sales",ascending=False)
      .reset_index()
)

product_performance.to_csv(
    OUTPUT_DIR/"product_performance.csv",
    index=False
)

# Phase 6 — Dashboard Filter Dataset

Dashboard interaktif membutuhkan filter agar pengguna dapat melakukan eksplorasi data berdasarkan kebutuhan tertentu.

Pada tahap ini dibuat dataset filter yang akan digunakan pada dashboard Streamlit, sehingga pilihan filter dapat dimuat secara dinamis tanpa melakukan proses ekstraksi ulang dari dataset utama.

Filter yang disiapkan meliputi:

- Year
- Region
- Segment
- Category
- Ship Mode
- State

Pendekatan ini meningkatkan efisiensi dashboard dan memudahkan pengembangan fitur interaktif.

In [57]:
import json

In [58]:
dashboard_filters = {

    "Year": sorted(df["Year"].unique().tolist()),

    "Region": sorted(df["Region"].unique().tolist()),

    "Segment": sorted(df["Segment"].unique().tolist()),

    "Category": sorted(df["Category"].unique().tolist()),

    "Ship Mode": sorted(df["Ship Mode"].unique().tolist()),

    "State": sorted(df["State"].unique().tolist())

}

dashboard_filters

{'Year': [2014, 2015, 2016, 2017],
 'Region': ['Central', 'East', 'South', 'West'],
 'Segment': ['Consumer', 'Corporate', 'Home Office'],
 'Category': ['Furniture', 'Office Supplies', 'Technology'],
 'Ship Mode': ['First Class', 'Same Day', 'Second Class', 'Standard Class'],
 'State': ['Alabama',
  'Arizona',
  'Arkansas',
  'California',
  'Colorado',
  'Connecticut',
  'Delaware',
  'District of Columbia',
  'Florida',
  'Georgia',
  'Idaho',
  'Illinois',
  'Indiana',
  'Iowa',
  'Kansas',
  'Kentucky',
  'Louisiana',
  'Maine',
  'Maryland',
  'Massachusetts',
  'Michigan',
  'Minnesota',
  'Mississippi',
  'Missouri',
  'Montana',
  'Nebraska',
  'Nevada',
  'New Hampshire',
  'New Jersey',
  'New Mexico',
  'New York',
  'North Carolina',
  'North Dakota',
  'Ohio',
  'Oklahoma',
  'Oregon',
  'Pennsylvania',
  'Rhode Island',
  'South Carolina',
  'South Dakota',
  'Tennessee',
  'Texas',
  'Utah',
  'Vermont',
  'Virginia',
  'Washington',
  'West Virginia',
  'Wisconsin',
  'W

In [59]:
with open(
    OUTPUT_DIR / "dashboard_filters.json",
    "w"
) as f:

    json.dump(
        dashboard_filters,
        f,
        indent=4
    )

In [60]:
with open(
    OUTPUT_DIR / "dashboard_filters.json"
) as f:

    filters = json.load(f)

filters

{'Year': [2014, 2015, 2016, 2017],
 'Region': ['Central', 'East', 'South', 'West'],
 'Segment': ['Consumer', 'Corporate', 'Home Office'],
 'Category': ['Furniture', 'Office Supplies', 'Technology'],
 'Ship Mode': ['First Class', 'Same Day', 'Second Class', 'Standard Class'],
 'State': ['Alabama',
  'Arizona',
  'Arkansas',
  'California',
  'Colorado',
  'Connecticut',
  'Delaware',
  'District of Columbia',
  'Florida',
  'Georgia',
  'Idaho',
  'Illinois',
  'Indiana',
  'Iowa',
  'Kansas',
  'Kentucky',
  'Louisiana',
  'Maine',
  'Maryland',
  'Massachusetts',
  'Michigan',
  'Minnesota',
  'Mississippi',
  'Missouri',
  'Montana',
  'Nebraska',
  'Nevada',
  'New Hampshire',
  'New Jersey',
  'New Mexico',
  'New York',
  'North Carolina',
  'North Dakota',
  'Ohio',
  'Oklahoma',
  'Oregon',
  'Pennsylvania',
  'Rhode Island',
  'South Carolina',
  'South Dakota',
  'Tennessee',
  'Texas',
  'Utah',
  'Vermont',
  'Virginia',
  'Washington',
  'West Virginia',
  'Wisconsin',
  'W

# Phase 7 — Final Validation & Export

Tahap terakhir bertujuan untuk memastikan seluruh dataset hasil transformasi berhasil dibuat dan siap digunakan pada aplikasi dashboard.

Pada tahap ini dilakukan:

- Validasi seluruh file hasil ekspor.
- Pemeriksaan ukuran dataset.
- Pemeriksaan tipe data.
- Menampilkan preview setiap dataset.
- Menyusun struktur folder akhir proyek.

Dengan proses validasi ini, pipeline data menjadi lebih terstruktur dan siap digunakan pada tahap pengembangan dashboard.

In [61]:
import os

files = sorted(os.listdir(OUTPUT_DIR))

print("Total Files :", len(files))

for file in files:
    print(file)

Total Files : 13
category_performance.csv
category_sales.csv
customer_performance.csv
customer_sales.csv
dashboard_filters.json
dashboard_kpi.csv
kpi.csv
monthly_sales.csv
product_performance.csv
product_sales.csv
region_performance.csv
region_sales.csv
sales_trend.csv


In [62]:
print("Dashboard KPI")
print(dashboard_kpi.shape)

print("\nMonthly Sales")
print(monthly_dashboard.shape)

print("\nCategory Performance")
print(category_performance.shape)

print("\nCustomer Performance")
print(customer_performance.shape)

print("\nProduct Performance")
print(product_performance.shape)

print("\nRegion Performance")
print(region_performance.shape)

Dashboard KPI
(1, 7)

Monthly Sales
(48, 6)

Category Performance
(3, 4)

Customer Performance
(793, 4)

Product Performance
(1850, 3)

Region Performance
(4, 3)


In [63]:
dashboard_kpi.head()

,Total Sales,Total Profit,Total Orders,Total Customers,Total Products,Average Order Value,Profit Margin (%)
0,2.297201e+06,286397.0217,5009,793,1862,458.614666,12.467217


In [64]:
monthly_dashboard.head()

,Year,Month,Month Name,Sales,Profit,Orders
0,2014,1,January,14236.895,2450.1907,32
1,2014,2,February,4519.892,862.3084,28
2,2014,3,March,55691.009,498.7299,71
3,2014,4,April,28295.345,3488.8352,66
4,2014,5,May,23648.287,2738.7096,69


In [65]:
category_performance.head()

,Category,Sales,Profit,Quantity
0,Furniture,741999.7953,18451.2728,8028
1,Office Supplies,719047.0320,122490.8008,22906
2,Technology,836154.0330,145454.9481,6939


In [66]:
customer_performance.head()

,Customer Name,Sales,Profit,Orders
0,Sean Miller,25043.050,-1980.7393,5
1,Tamara Chand,19052.218,8981.3239,5
2,Raymond Buch,15117.339,6976.0959,6
3,Tom Ashbrook,14595.620,4703.7883,4
4,Adrian Barton,14473.571,5444.8055,10


In [67]:
product_performance.head()

,Product Name,Sales,Profit
0,Canon imageCLASS 2200 Advanced Copier,61599.824,2.519993e+04
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384,7.753039e+03
2,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1.811078e+03
3,HON 5400 Series Task Chairs for Big and Tall,21870.576,5.684342e-14
4,GBC DocuBind TL300 Electric Binding System,19823.479,2.233505e+03


In [68]:
region_performance.head()

,Region,Sales,Profit
0,Central,501239.8908,39706.3625
1,East,678781.2400,91522.7800
2,South,391721.9050,46749.4303
3,West,725457.8245,108418.4489


In [69]:
datasets = {
    "dashboard_kpi": dashboard_kpi,
    "monthly_dashboard": monthly_dashboard,
    "category_performance": category_performance,
    "customer_performance": customer_performance,
    "product_performance": product_performance,
    "region_performance": region_performance,
}

for name, data in datasets.items():
    print("=" * 50)
    print(name)
    print(data.isnull().sum())

dashboard_kpi
Total Sales            0
Total Profit           0
Total Orders           0
Total Customers        0
Total Products         0
Average Order Value    0
Profit Margin (%)      0
dtype: int64
monthly_dashboard
Year          0
Month         0
Month Name    0
Sales         0
Profit        0
Orders        0
dtype: int64
category_performance
Category    0
Sales       0
Profit      0
Quantity    0
dtype: int64
customer_performance
Customer Name    0
Sales            0
Profit           0
Orders           0
dtype: int64
product_performance
Product Name    0
Sales           0
Profit          0
dtype: int64
region_performance
Region    0
Sales     0
Profit    0
dtype: int64


In [70]:
summary = pd.DataFrame({

    "Dataset":[
        "Dashboard KPI",
        "Monthly Sales",
        "Category Performance",
        "Customer Performance",
        "Product Performance",
        "Region Performance"
    ],

    "Rows":[
        len(dashboard_kpi),
        len(monthly_dashboard),
        len(category_performance),
        len(customer_performance),
        len(product_performance),
        len(region_performance)
    ],

    "Columns":[
        dashboard_kpi.shape[1],
        monthly_dashboard.shape[1],
        category_performance.shape[1],
        customer_performance.shape[1],
        product_performance.shape[1],
        region_performance.shape[1]
    ]

})

summary

,Dataset,Rows,Columns
0,Dashboard KPI,1,7
1,Monthly Sales,48,6
2,Category Performance,3,4
3,Customer Performance,793,4
4,Product Performance,1850,3
5,Region Performance,4,3


# Conclusion

Pada notebook ini telah dilakukan proses persiapan data untuk dashboard analitik penjualan.

Seluruh dataset telah ditransformasikan menjadi beberapa tabel agregasi yang lebih ringan dan efisien untuk digunakan pada aplikasi dashboard berbasis Streamlit.

Output utama notebook ini meliputi:

- Dashboard KPI
- Sales Trend
- Category Performance
- Customer Performance
- Product Performance
- Region Performance
- Dashboard Filters

Dengan pemisahan proses transformasi data dan visualisasi, dashboard dapat berjalan lebih cepat serta mengikuti praktik pengembangan data pipeline yang umum digunakan di industri.